# TimesFM Zero-Shot Weather Forecasting

This notebook evaluates [Google's TimesFM](https://github.com/google-research/timesfm) — a pretrained time-series foundation model — against your Zephyr weather data.

**No training required.** TimesFM forecasts zero-shot, so we skip straight to evaluation.

### What we'll do
1. Load data from the SQLite database
2. Load the pretrained TimesFM 2.5 model (~800 MB, cached after first download)
3. Run a **persistence baseline** (predict = last known value)
4. Run a **TimesFM backtest** over the held-out 20% of history
5. Compare results and visualise errors
6. Generate a **current 1-hour forecast** with uncertainty bands

## 1. Install & imports

In [ ]:
# Install timesfm if not already present
# (tsai and other deps are assumed to already be installed via uv sync)
import importlib
if importlib.util.find_spec("timesfm") is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timesfm"])
    print("timesfm installed.")
else:
    print("timesfm already installed.")

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make sure modelling/ is on the path so we can import train.py and timesfm_forecast.py
_HERE = Path(".").resolve()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

from timesfm_forecast import (
    CONTEXT_LEN,
    FEATURES,
    HORIZON,
    backtest,
    forecast_current,
    load_model,
    persistence_backtest,
    print_backtest_comparison,
)
from train import get_training_data

plt.rcParams["figure.dpi"] = 120
print("Imports OK")

## 2. Configuration

In [ ]:
# ── geographic area ────────────────────────────────────────────────────────────
STATION_NAME = "Coronet Tandems"  # centre station (must match stations table exactly)
MAX_DISTANCE = 20                  # radius in km

# ── forecast settings ──────────────────────────────────────────────────────────
CONTEXT_LEN  = 60    # past timesteps given to the model (10 hours)
HORIZON      = 6     # steps to forecast (1 hour at 10-min intervals)
STRIDE       = 6     # gap between backtest evaluation points (non-overlapping)
TEST_FRAC    = 0.2   # fraction of history held out for evaluation

print(f"Area : stations within {MAX_DISTANCE} km of '{STATION_NAME}'")
print(f"Input: {CONTEXT_LEN} timesteps ({CONTEXT_LEN * 10 / 60:.0f} hours of history)")
print(f"Output: {HORIZON} timesteps ({HORIZON * 10} minutes ahead)")

## 3. Load data

In [ ]:
# The database path is relative to the project root, one level up from modelling/
import os
os.chdir(_HERE.parent)  # run from project root so train.py can find zephyr-model.db

df = get_training_data(STATION_NAME, MAX_DISTANCE)

print(f"Rows     : {len(df):,}")
print(f"Stations : {df['station_id'].nunique()}")
print(f"Columns  : {df.columns.tolist()}")
df.head(3)

In [ ]:
# Quick look at the data distributions
df[FEATURES].describe().round(2)

## 4. Load TimesFM model

First run downloads ~800 MB from HuggingFace. Subsequent runs use the local cache.

In [ ]:
print("Loading TimesFM 2.5 (200M)...")
model = load_model()
print("Model ready.")

## 5. Persistence baseline

The simplest possible forecast: predict that the next N steps equal the last observed value.
This is the minimum bar TimesFM needs to beat.

In [ ]:
print("Running persistence baseline...")
pers = persistence_backtest(df, horizon=HORIZON, stride=STRIDE, test_fraction=TEST_FRAC)

print("\nPersistence MAE:")
for var, mae in pers["mae"].items():
    print(f"  {var:<22} {mae:.3f}")

## 6. TimesFM backtest

For each station × variable, slide a context window through the test portion of the
history and compare TimesFM's forecasts to actual observations.

In [ ]:
print("Running TimesFM backtest (may take a few minutes)...")
tfm = backtest(df, model, context_len=CONTEXT_LEN, horizon=HORIZON,
               stride=STRIDE, test_fraction=TEST_FRAC)

print(f"\nBacktest records: {len(tfm['records']):,}")
tfm["records"].head()

## 7. Results comparison

In [ ]:
print_backtest_comparison(tfm, pers)

In [ ]:
# Side-by-side bar chart
fig, ax = plt.subplots(figsize=(9, 4))

x = np.arange(len(FEATURES))
w = 0.35
pers_vals = [pers["mae"].get(v, 0) for v in FEATURES]
tfm_vals  = [tfm["mae"].get(v, 0)  for v in FEATURES]

ax.bar(x - w/2, pers_vals, w, label="Persistence", color="#d62728", alpha=0.8)
ax.bar(x + w/2, tfm_vals,  w, label="TimesFM 2.5", color="#1f77b4", alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(FEATURES)
ax.set_ylabel("MAE")
ax.set_title("Backtest MAE — TimesFM vs Persistence Baseline")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### MAE by forecast horizon step

Does accuracy degrade as we look further ahead? (Step 1 = 10 min, step 6 = 60 min)

In [ ]:
fig, axes = plt.subplots(1, len(FEATURES), figsize=(14, 3.5), sharey=False)

for ax, var in zip(axes, FEATURES):
    tfm_by_step  = (tfm["records"][tfm["records"]["variable"] == var]
                    .groupby("horizon_step")["abs_error"].mean())
    pers_by_step = (pers["records"][pers["records"]["variable"] == var]
                    .groupby("horizon_step")["abs_error"].mean())

    steps = tfm_by_step.index
    ax.plot(steps, pers_by_step.reindex(steps), "o--", color="#d62728",
            label="Persistence", linewidth=1.5)
    ax.plot(steps, tfm_by_step, "o-",  color="#1f77b4",
            label="TimesFM",    linewidth=1.5)
    ax.set_title(var)
    ax.set_xlabel("Horizon step")
    ax.set_ylabel("MAE")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xticks(steps)

fig.suptitle("MAE by Forecast Horizon Step (step 1 = +10 min, step 6 = +60 min)",
             fontsize=11)
plt.tight_layout()
plt.show()

### Sample forecast traces

Pick a station and variable to plot actual vs predicted over the test window.

In [ ]:
PLOT_STATION = tfm["records"]["station_id"].iloc[0]   # change to any station ID
PLOT_VAR     = "temperature"                            # change to any FEATURES variable
N_SAMPLES    = 200                                      # number of eval points to show

rec = (
    tfm["records"]
    [(tfm["records"]["station_id"] == PLOT_STATION)
     & (tfm["records"]["variable"]   == PLOT_VAR)
     & (tfm["records"]["horizon_step"] == 1)]           # step-1 = next 10 minutes
    .tail(N_SAMPLES)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rec.index, rec["actual"],    label="Actual",     linewidth=1.2)
ax.plot(rec.index, rec["predicted"], label="TimesFM",    linewidth=1.0, alpha=0.85)
ax.set_xlabel("Evaluation point")
ax.set_ylabel(PLOT_VAR)
ax.set_title(f"{PLOT_VAR} — Station {PLOT_STATION}  (step 1, last {N_SAMPLES} evals)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Error distribution

In [ ]:
fig, axes = plt.subplots(1, len(FEATURES), figsize=(14, 3.5))

for ax, var in zip(axes, FEATURES):
    errors = (
        tfm["records"]
        [(tfm["records"]["variable"] == var) & (tfm["records"]["horizon_step"] == 1)]
        ["error"]
        .dropna()
    )
    ax.hist(errors, bins=60, color="#1f77b4", alpha=0.75, edgecolor="none")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"{var}\n(bias={errors.mean():.3f})")
    ax.set_xlabel("Prediction error")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)

fig.suptitle("Step-1 Forecast Error Distribution (TimesFM)", fontsize=11)
plt.tight_layout()
plt.show()

## 8. Current 1-hour forecast

Uses the most recent `CONTEXT_LEN` observations from each station to produce the
next 6 × 10-minute forecasts, with a 10th–90th percentile uncertainty band.

In [ ]:
print("Generating current forecasts...")
forecasts = forecast_current(df, model, context_len=CONTEXT_LEN, horizon=HORIZON)
print(f"Forecast rows: {len(forecasts)}")
forecasts.head(12)

In [ ]:
# Plot all variables for the first station
FORECAST_STATION = forecasts["station_id"].iloc[0]
sdf = forecasts[forecasts["station_id"] == FORECAST_STATION]

fig, axes = plt.subplots(1, len(FEATURES), figsize=(14, 3.5))
minutes = [str(s * 10) for s in sdf["horizon_step"].unique()]

for ax, var in zip(axes, FEATURES):
    vdf = sdf[sdf["variable"] == var].reset_index(drop=True)
    steps = vdf["horizon_step"]

    ax.fill_between(steps, vdf["q10"], vdf["q90"],
                    alpha=0.25, color="#1f77b4", label="10–90% band")
    ax.plot(steps, vdf["q50"],       "--", color="#1f77b4", linewidth=1.2, label="Median")
    ax.plot(steps, vdf["predicted"], "-o", color="#ff7f0e", linewidth=1.5, label="Point forecast")

    ax.set_title(var)
    ax.set_xlabel("Horizon step (+10 min each)")
    ax.set_xticks(steps)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(f"1-Hour Forecast — Station {FORECAST_STATION}", fontsize=11)
plt.tight_layout()
plt.show()

## 9. Longer context experiment

TimesFM supports up to 16 384 context steps. Try a full 24-hour context window
(144 steps at 10-min intervals) to see if more history helps.

In [ ]:
LONG_CONTEXT = 144  # 24 hours

print(f"Running TimesFM backtest with context_len={LONG_CONTEXT}...")
tfm_long = backtest(df, model, context_len=LONG_CONTEXT, horizon=HORIZON,
                    stride=STRIDE, test_fraction=TEST_FRAC)

# Compare short vs long context
print(f"\n{'Variable':<22} {'ctx=60 MAE':>12} {'ctx=144 MAE':>13} {'Δ':>10}")
print("-" * 60)
for var in FEATURES:
    short = tfm["mae"].get(var, float("nan"))
    long_ = tfm_long["mae"].get(var, float("nan"))
    if not (np.isnan(short) or np.isnan(long_)) and short > 0:
        delta = (short - long_) / short * 100
        arrow = "▲" if delta > 0 else "▼"
        print(f"  {var:<20} {short:>12.3f} {long_:>13.3f} {arrow}{abs(delta):>8.1f}%")
    else:
        print(f"  {var:<20} {short:>12.3f} {long_:>13.3f}")
print("\n▲ = longer context better")